# LexIQ — Data Preprocessing


## 1. Load Data & Initialize Tokenizer


In [1]:
import pandas as pd
from transformers import AutoTokenizer

data = pd.read_parquet("../data/processed/df_flat.parquet")


/Users/abcd/PycharmProjects/LexIQ/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [3]:
tokenizer.vocab_size

30522

In [4]:
data[data['is_impossible'] == False][['question', 'answer_text']].iloc[0]

question       Highlight the parts (if any) of this contract ...
answer_text                                DISTRIBUTOR AGREEMENT
Name: 0, dtype: str

## 2. Load Contract Texts
We load full contract texts from raw TXT files and merge them with the QA pairs by contract title.

In [5]:
from pathlib import Path

contracts_folder = Path('../data/raw/full_contract_txt')

contracts = [
    {'title' :  f.stem, 'text': f.read_text(encoding='utf8')} for f in contracts_folder.glob('*.txt')
]

contracts = pd.DataFrame(contracts)

df = data.merge(
    contracts,
    on='title',
    how='left',

)

In [6]:
df.shape

(20910, 7)

In [8]:
from sklearn.model_selection import train_test_split

titles = df['title'].unique().tolist()
train_titles, test_titles = train_test_split(titles, test_size=0.2, random_state=42)

In [11]:
df_train = df[df['title'].isin(train_titles)]
df_test = df[df['title'].isin(test_titles)]


In [12]:
print(df_train.shape)
print(df_test.shape)

(16728, 7)
(4182, 7)


## 3. Tokenization — Single Example
We test the tokenizer on a single example to verify offset mapping behavior before processing the full dataset.

In [13]:
example_row = row = df[df['is_impossible'] == False].iloc[0]

sample = tokenizer(example_row['question'],
                   example_row['text'],
                   max_length=512,
                   return_offsets_mapping=True)
print(sample.keys())
print(len(sample["input_ids"]))

KeysView({'input_ids': [101, 12944, 1996, 3033, 1006, 2065, 2151, 1007, 1997, 2023, 3206, 3141, 2000, 1000, 6254, 2171, 1000, 2008, 2323, 2022, 8182, 2011, 1037, 5160, 1012, 4751, 1024, 1996, 2171, 1997, 1996, 3206, 102, 8327, 2184, 1012, 1020, 16632, 3820, 2023, 16632, 3820, 1006, 1996, 1000, 3820, 1000, 1007, 2003, 2081, 2011, 1998, 2090, 3751, 2103, 13058, 1012, 1010, 1037, 8452, 3840, 1006, 1000, 2194, 1000, 1007, 1998, 3751, 2103, 1997, 4307, 11775, 1006, 1000, 16632, 1000, 1007, 2023, 5504, 2154, 1997, 2244, 1010, 2639, 1012, 25521, 2015, 1037, 1012, 1996, 2194, 1005, 1055, 2449, 1012, 1996, 2194, 2003, 12825, 5117, 1999, 1996, 2449, 1997, 4855, 2019, 2943, 8122, 5080, 1010, 2029, 2003, 3615, 2000, 2004, 2019, 1000, 2943, 3828, 2099, 1000, 2029, 2089, 2022, 5301, 2030, 4728, 2904, 2013, 2049, 2556, 5512, 1006, 1996, 1000, 3688, 1000, 1007, 1012, 1996, 2194, 2089, 8526, 1999, 1996, 2449, 1997, 4855, 2060, 3688, 2030, 2060, 5733, 2060, 2084, 1996, 3688, 1010, 2029, 2097, 2022, 2641

In [14]:
sample['offset_mapping'][:10]

[(0, 0),
 (0, 9),
 (10, 13),
 (14, 19),
 (20, 21),
 (21, 23),
 (24, 27),
 (27, 28),
 (29, 31),
 (32, 36)]


## 4. Span Mapping Logic
DistilBERT operates on tokens, not characters. We use offset_mapping and sequence_ids() to locate the answer span within the context (sequence 1), ignoring question tokens (sequence 0)

In [15]:
for i, j in zip(sample['offset_mapping'], sample.sequence_ids()):
    if i[0] <= 44 and i[1] > 44 and j == 1:
        print(i[0], i[1], j)


44 55 1


## 5. Tokenize & Find Positions — Full Dataset
We apply tokenize_and_find_positions() across all 20,910 rows. For impossible examples, positions default to (0, 0). For truncated answers beyond 512 tokens, positions also fall back to (0, 0).

In [16]:
def tokenize_and_find_positions(row):
    encoding = tokenizer(
        row['question'],
        row['text'],
        max_length=512,
        truncation=True,
        return_offsets_mapping=True
    )

    if row['is_impossible']:
        start_position = 0
        end_position = 0
    else:
        start_position = 0
        end_position = 0
        answer_start = row['answer_start']
        answer_end = answer_start + len(row['answer_text'])

        for idx, (offsets, seq_id) in enumerate(zip(encoding['offset_mapping'], encoding.sequence_ids())):
            if seq_id != 1:
                continue
            if offsets[0] <= answer_start < offsets[1]:
                start_position = idx
            if offsets[0] < answer_end <= offsets[1]:
                end_position = idx

    return {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'start_position': start_position,
        'end_position': end_position
    }

In [17]:
result = tokenize_and_find_positions(df[df['is_impossible'] == False].iloc[0])
print(result['start_position'], result['end_position'])

37 38


In [18]:
df['processed'] = df.apply(tokenize_and_find_positions, axis=1)


In [19]:
df_train['processed'] = df_train.apply(tokenize_and_find_positions, axis=1)
df_test['processed'] = df_test.apply(tokenize_and_find_positions, axis=1)

In [42]:
df.head()

,title,clause_type,question,is_impossible,answer_text,answer_start,text,processed
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,Highlight the parts (if any) of this contract ...,False,DISTRIBUTOR AGREEMENT,44.0,EXHIBIT 10.6\n\n ...,"{'input_ids': [101, 12944, 1996, 3033, 1006, 2..."
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Highlight the parts (if any) of this contract ...,False,Distributor,244.0,EXHIBIT 10.6\n\n ...,"{'input_ids': [101, 12944, 1996, 3033, 1006, 2..."
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Agreement Date,Highlight the parts (if any) of this contract ...,False,"7th day of September, 1999.",263.0,EXHIBIT 10.6\n\n ...,"{'input_ids': [101, 12944, 1996, 3033, 1006, 2..."
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Effective Date,Highlight the parts (if any) of this contract ...,False,The term of this Agreement shall be ten (10)...,5268.0,EXHIBIT 10.6\n\n ...,"{'input_ids': [101, 12944, 1996, 3033, 1006, 2..."
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Expiration Date,Highlight the parts (if any) of this contract ...,False,The term of this Agreement shall be ten (10)...,5268.0,EXHIBIT 10.6\n\n ...,"{'input_ids': [101, 12944, 1996, 3033, 1006, 2..."


## 6. Save Processed Dataset
Tokenized dataset saved to data/processed/df_processed.parquet for use in fine-tuning notebook (Colab).

In [20]:
df[['title', 'clause_type', 'question', 'is_impossible', 'processed']].to_parquet('../data/processed/df_processed.parquet', index=False)

In [21]:
df_train[['title', 'clause_type', 'question', 'is_impossible', 'processed']].to_parquet('../data/processed/df_train.parquet', index=False)
df_test[['title', 'clause_type', 'question', 'is_impossible', 'processed']].to_parquet('../data/processed/df_test.parquet', index=False)